## Data Ingestion

In [1]:
import os
%pwd

'd:\\Data Science\\END to END Proj\\Introvert vs Extrovert\\Introvert-Vs-Extrovert\\research'

In [2]:
os.chdir("../")

In [3]:
## ENTITY
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:Path
    source_URL:str
    local_data_file:Path
    unzip_dir:Path

In [4]:
from src.IntrovertVsExtrovert.constant import *
from src.IntrovertVsExtrovert.utils.common import read_yaml,create_directories 

In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [6]:
import os
import urllib.request as request
import zipfile
import pandas as pd
from IntrovertVsExtrovert import logger
from IntrovertVsExtrovert.utils.common import get_size

In [7]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        """Downloads the file from source_URL to local_data_file"""
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(self.config.local_data_file), exist_ok=True)
        
        if not os.path.exists(self.config.local_data_file):
            logger.info(f"Downloading data from {self.config.source_URL}...")
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,  # Use source_URL here
                filename=self.config.local_data_file  # Save to local_data_file
            )
            logger.info(f"Download completed to: {filename}")
            logger.debug(f"Download headers: {headers}")
        else:
            logger.info(f"File already exists at {self.config.local_data_file}, size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        """Extracts the downloaded zip file to unzip_dir"""
        logger.info(f"Extracting zip file from {self.config.local_data_file} to {self.config.unzip_dir}")
        
        # Create extraction directory if it doesn't exist
        os.makedirs(self.config.unzip_dir, exist_ok=True)
        
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(self.config.unzip_dir)

        print("✅ Listing files after unzip:")
        for root, dirs, files in os.walk(self.config.unzip_dir):
            for file in files:
                print(os.path.join(root, file))

        
        logger.info(f"Successfully extracted to {self.config.unzip_dir}")

    def load_datasets(self):
        """Load train.csv and org CSVs from the ExtvsInt folder inside unzip_dir"""
        base_dir = os.path.join(self.config.unzip_dir, "ExtvsInt")   # <- key change

        train_df = pd.read_csv(os.path.join(base_dir, "train.csv"))
        org1     = pd.read_csv(os.path.join(base_dir, "personality_dataset.csv"))
        org2     = pd.read_csv(os.path.join(base_dir, "personality_dataset_1.csv"))
        org3     = pd.read_csv(os.path.join(base_dir, "personality_dataset_2.csv"))

        org_combined = pd.concat([org1, org2], ignore_index=True)
        # inside load_datasets() ─ after org_combined is created
        save_path = os.path.join(self.config.unzip_dir, "org_combined.csv")
        org_combined.to_csv(save_path, index=False)
        logger.info(f"org_combined saved to {save_path}")

        return train_df, org_combined



In [8]:

try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)

    # Step 1: Download
    data_ingestion.download_file()

    # Step 2: Unzip
    data_ingestion.extract_zip_file()

    # Step 3: Load CSVs from extracted folder
    train_df, org_combined = data_ingestion.load_datasets()

    print("Train Shape:", train_df.shape)
    print("Org Combined Shape:", org_combined.shape)

except Exception as e:
    raise e


[2025-07-11 17:19:44,245: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-07-11 17:19:44,249: INFO: common: yaml file: params.yaml loaded successfully]
[2025-07-11 17:19:44,252: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-07-11 17:19:44,254: INFO: common: created directory at: artifacts]
[2025-07-11 17:19:44,259: INFO: common: created directory at: artifacts/data_ingestion]
[2025-07-11 17:19:44,263: INFO: 3203101843: Downloading data from https://github.com/gowtham-dd/Datasets/raw/main/ExtvsInt.zip...]
[2025-07-11 17:19:45,390: INFO: 3203101843: Download completed to: artifacts/data_ingestion/data.zip]
[2025-07-11 17:19:45,411: INFO: 3203101843: Extracting zip file from artifacts/data_ingestion/data.zip to artifacts/data_ingestion]
✅ Listing files after unzip:
artifacts/data_ingestion\data.zip
artifacts/data_ingestion\ExtvsInt\personality_dataset.csv
artifacts/data_ingestion\ExtvsInt\personality_dataset_1.csv
artifacts/data_ingestion\ExtvsInt\